# Avance Semana 2 — SROIE + WildReceipt
Jesús López y Sebastián Rojas · Proyecto de extracción de facturas y comprobantes.

Este notebook ejecuta el código modular del repositorio. Conserva SROIE + WildReceipt como base principal y Tesseract frente a Document AI como comparación prevista. No entrena una red ni sustituye la segunda solución por Random Forest.

**Las salidas de este notebook se generan al ejecutarlo.** Los resultados ya medidos durante la preparación están en `reports/AVANCE_EJECUTADO.md`; no se insertan aquí como si hubieran sido ejecutados en tu equipo.

Orden: configurar entorno → colocar bases → crear manifiesto → EDA → baseline → revisión → API opcional. No correr todo automáticamente sin configurar rutas.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json
# Cambiar si el notebook se abrió desde otra carpeta.
PROJECT = Path.cwd()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
# En Colab, descomprimir el ZIP y asignar aquí su carpeta raíz.
# PROJECT = Path("/content/proyecto_ocr")
assert (PROJECT / "pyproject.toml").exists(), "Ajusta PROJECT a la carpeta descomprimida"
os.chdir(PROJECT)
print("Proyecto:", PROJECT)


## 1. Instalación (solo la primera vez)
En Colab, poner `INSTALL=True`. En Windows instalar primero el ejecutable Tesseract según el README. Esta celda instala bibliotecas, no descarga bases ni llama a Google Cloud.


In [ ]:
INSTALL = False
if INSTALL:
    if "google.colab" in sys.modules:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "tesseract-ocr", "tesseract-ocr-eng", "tesseract-ocr-spa"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[notebook]"], check=True)

def cli(*args):
    subprocess.run([sys.executable, "-m", "lector_facturas", *map(str,args)], check=True)

cli("doctor")


## 2. Datos
Usar ambas bases. Descarga manual o `scripts/download_data.py` según README; verificar condiciones. Los archivos originales no vienen dentro del ZIP. La raíz de SROIE contiene `train/img`, `train/box` y `train/entities`; la de WildReceipt contiene `train.txt`, `test.txt` y `class_list.txt`.


In [ ]:
SROIE = Path("data/raw/sroie_distribution/SROIE2019")
WILD = Path("data/raw/wildreceipt")
assert (SROIE / "train/img").exists(), "Falta SROIE o debes corregir la ruta"
assert (WILD / "class_list.txt").exists(), "Falta WildReceipt o debes corregir la ruta"
cli("prepare", "--sroie", SROIE, "--wildreceipt", WILD)
cli("eda")


## 3. EDA ejecutado
La cobertura es disponibilidad de anotaciones, no exactitud. Revisar incidencias y diferencias por dataset antes de interpretar resultados. El test oficial está reservado.


In [ ]:
import pandas as pd
from IPython.display import display, Markdown, Image
summary = json.loads(Path("outputs/eda/summary.json").read_text(encoding="utf-8"))
display(summary)
display(pd.read_csv("outputs/eda/coverage.csv"))
display(pd.read_csv("outputs/eda/incidents.csv").head(30))
display(Image(filename="outputs/eda/coverage.png"))


## 4. Primer experimento de extracción
Semilla, normalización y OCR en `configs/default.json`. No se ajusta nada sobre test. Se procesan hasta 20 documentos por dataset de la validación de desarrollo. Usar un nombre nuevo por ejecución. El conjunto final de ≥30 documentos propios es una etapa distinta.


In [ ]:
RUN = Path("outputs/baseline_mi_equipo_v1")
cli("run", "--engine", "tesseract", "--split", "validation", "--limit-per-dataset", 20, "--out", RUN)


In [ ]:
metrics = pd.read_csv(RUN / "evaluation/metrics.csv")
display(metrics)
display(json.loads((RUN / "evaluation/summary.json").read_text(encoding="utf-8")))
errors = pd.read_csv(RUN / "evaluation/errors.csv")
display(errors[errors["correct"] == 0].head(20))
display(Markdown((RUN / "evaluation/RESULTADOS.md").read_text(encoding="utf-8")))


## 5. Document AI — ejecución opcional con cuenta configurada
Consultar `docs/DOCUMENT_AI.md`. No afirmar resultados de Google si no se ejecutó la API. Usar el mismo manifiesto y límite. La llamada envía documentos a Google y puede generar cargos. No poner claves dentro del notebook.


In [ ]:
RUN_GOOGLE = False
GOOGLE_RUN = Path("outputs/documentai_mi_equipo_v1")
if RUN_GOOGLE:
    assert os.environ.get("DOCUMENTAI_PROCESSOR_RESOURCE"), "Configura procesador y credenciales ADC"
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[cloud]"], check=True)
    cli("run", "--engine", "google", "--allow-cloud", "--split", "validation", "--limit-per-dataset", 20, "--out", GOOGLE_RUN)
    cli("compare", "--a", RUN, "--b", GOOGLE_RUN, "--out", "outputs/comparacion_mi_equipo_v1")
    display(pd.read_csv("outputs/comparacion_mi_equipo_v1/comparison.csv"))


## 6. Párrafo del avance (completar con los resultados de TU ejecución)
- Fuentes y cantidades efectivamente descargadas:
- Documentos de desarrollo/validación y test reservado:
- Dos hallazgos EDA con cifras y ruta de evidencia:
- Exactitud/F1 por campo y dataset, con soporte:
- Fallos de ejecución y latencia sin caché:
- Limitaciones de anotaciones y muestra:
- Estado real de la comparación con Document AI:
- Siguiente modificación justificada por los errores de desarrollo:

Guardar el notebook con salidas para conservar evidencia. La ficha adicional sigue teniendo máximo una página; el código y estas tablas pertenecen al repositorio/material de apoyo.
